# PINN Vancomycin — Sensitivity Analysis (Kaggle, 2×T4)

Notebook za pokretanje glavnog eksperimenta na Kaggle GPU sesiji.

**Ključne razlike u odnosu na Colab verziju:**
- Nema Google Drive-a → rezultati idu u `/kaggle/working/` (automatski dostupni za download)
- 2×T4 GPU: `cuda:0` i `cuda:1` — 1-odeljni i 2-odeljni model rade **paralelno** (threading)
- Kaggle sesija traje do **12h** — dovoljno za oba modela paralelno

**Podešavanja pre pokretanja:**
1. `Settings → Accelerator → GPU T4 x2`
2. `Settings → Persistence → Variables and Files` (čuva `/kaggle/working/` između sesija)
3. U ćeliji 2: postavi `REPO_URL` na tvoj GitHub link

---
| Ćelija | Akcija | Trajanje |
|--------|--------|----------|
| 1 | GPU check + putanje | 5s |
| 2 | Kloniranje repoa + instalacija | 1–2 min |
| 3 | Generisanje podataka | 30s |
| 4 | Import modula | 5s |
| 5 | Test run (provjera) | ~5 min |
| 6 | **Paralelni eksperiment** (oba modela, oba GPU-a) | ~3–5h |
| 7 | Provjera napretka | 5s |
| 8 | Restore checkpoint | 10s |
| — | *(split sekcija — vidi dolje)* | — |

## Ćelija 1 — GPU check i putanje

In [ ]:
import torch
from pathlib import Path

n_gpu = torch.cuda.device_count()
print(f'PyTorch: {torch.__version__}')
print(f'CUDA dostupna: {torch.cuda.is_available()}')
print(f'Broj GPU-a: {n_gpu}')
for i in range(n_gpu):
    print(f'  cuda:{i} → {torch.cuda.get_device_name(i)}')

if n_gpu < 2:
    print('\nUPOZORENJE: manje od 2 GPU-a. Provjeri Settings → Accelerator → GPU T4 x2.')

OUT_DIR  = Path('/kaggle/working/pinn_results')
REPO_DIR = Path('/kaggle/working/pinn-vancomycin')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'\nIzlazni folder: {OUT_DIR}')
print(f'Repo folder:    {REPO_DIR}')

## Ćelija 2 — Kloniranje repoa i instalacija

⚠️ **Izmijeni `REPO_URL`** na tvoj GitHub link.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'   # <-- IZMIJENI OVO
REPO_DIR = Path('/kaggle/working/pinn-vancomycin')

if not REPO_DIR.exists():
    !git clone $REPO_URL {REPO_DIR}
else:
    print('Repo vec postoji, povlacim promjene...')
    !git -C {REPO_DIR} pull

!git -C {REPO_DIR} log --oneline -3
!pip install scipy seaborn -q
print('\nInstalacija gotova.')

## Ćelija 3 — Generisanje sintetičkih podataka

In [ ]:
!python {REPO_DIR}/src/data_processing.py

## Ćelija 4 — Import modula i konfiguracija

In [ ]:
import sys
import importlib.util
import torch
from pathlib import Path

REPO_DIR = Path('/kaggle/working/pinn-vancomycin')
OUT_DIR  = Path('/kaggle/working/pinn_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_DIR / 'src'))

def _load(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

sens = _load(REPO_DIR / 'experiments' / '03_sensitivity_analysis.py', 'sensitivity')

n_gpu   = torch.cuda.device_count()
DEVICE0 = 'cuda:0' if n_gpu >= 1 else 'cpu'
DEVICE1 = 'cuda:1' if n_gpu >= 2 else DEVICE0

print(f'1-odeljni model → {DEVICE0}')
print(f'2-odeljni model → {DEVICE1}')
print(f'Checkpoint folder: {OUT_DIR}')
print('Moduli učitani.')

## Ćelija 5 — Test run

Pokreće 8 kombinacija sa minimalnim brojem epoha (~5 min). Preporučeno pri prvom pokretanju.

In [ ]:
import threading, shutil

TEST_DIR = OUT_DIR / 'test'
TEST_DIR.mkdir(parents=True, exist_ok=True)
errors = []

def _test_1(): 
    try: sens.run_sensitivity_1comp(test_mode=True, out_dir=TEST_DIR, device=DEVICE0)
    except Exception as e: errors.append(f'1comp: {e}')

def _test_2():
    try: sens.run_sensitivity_2comp(test_mode=True, out_dir=TEST_DIR, device=DEVICE1)
    except Exception as e: errors.append(f'2comp: {e}')

t1, t2 = threading.Thread(target=_test_1), threading.Thread(target=_test_2)
t1.start(); t2.start()
t1.join();  t2.join()

if errors:
    for e in errors: print(f'GRESKA: {e}')
else:
    shutil.rmtree(TEST_DIR, ignore_errors=True)
    print('Test OK. Spreman za pravi eksperiment.')

## Ćelija 6 — Paralelni eksperiment (oba modela, oba GPU-a)

**1200 runova ukupno.** Procijenjeno trajanje (2×T4): **~3–5h** paralelno.

In [ ]:
import threading

errors = []

def _run_1comp():
    try:
        print(f'[1-comp] Start na {DEVICE0}')
        sens.run_sensitivity_1comp(test_mode=False, out_dir=OUT_DIR, device=DEVICE0)
        print('[1-comp] ZAVRSENO')
    except Exception as e:
        errors.append(f'1comp: {e}'); print(f'[1-comp] GRESKA: {e}')

def _run_2comp():
    try:
        print(f'[2-comp] Start na {DEVICE1}')
        sens.run_sensitivity_2comp(test_mode=False, out_dir=OUT_DIR, device=DEVICE1)
        print('[2-comp] ZAVRSENO')
    except Exception as e:
        errors.append(f'2comp: {e}'); print(f'[2-comp] GRESKA: {e}')

t1 = threading.Thread(target=_run_1comp)
t2 = threading.Thread(target=_run_2comp)
t1.start(); t2.start()
t1.join();  t2.join()

if errors:
    for e in errors: print(f'GRESKA: {e}')
else:
    print('Oba modela zavrsena. Rezultati su u Output tabu.')

## Ćelija 7 — Provjera napretka

Pokrenuti u bilo kom trenutku dok eksperiment radi.

In [ ]:
import pandas as pd
from pathlib import Path

OUT_DIR = Path('/kaggle/working/pinn_results')
TOTAL   = 5 * 4 * 30

for name in ['sensitivity_1comp.csv', 'sensitivity_2comp.csv']:
    p = OUT_DIR / name
    if not p.exists():
        print(f'{name}: ne postoji jos'); continue
    df = pd.read_csv(p)
    ok = df[df['status'] == 'ok']
    print(f'{name}: {len(df)}/{TOTAL} ({len(df)/TOTAL*100:.1f}%)  uspjesnih={len(ok)}')
    if len(ok) > 0:
        print(f'  Median PINN err_k10: {ok["pinn_err_k10"].median():.2f}%')
        print(f'  Po N: {ok["N"].value_counts().sort_index().to_dict()}')
    print()

## Ćelija 8 — Restore checkpoint

Koristiti samo ako se sesija prekinula. Priloži prethodni CSV kao Kaggle Dataset  
(`Settings → Add data`), izmijeni `INPUT_DIR` ispod, pa pokreni ćelije 1–4 → ova ćelija → ćelija 6.

In [ ]:
import shutil
from pathlib import Path

OUT_DIR   = Path('/kaggle/working/pinn_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR = Path('/kaggle/input/pinn-checkpoint')   # <-- izmijeni naziv dataseta

if not INPUT_DIR.exists():
    print(f'Dataset nije priložen: {INPUT_DIR}')
else:
    for csv in sorted(INPUT_DIR.glob('sensitivity_*.csv')):
        dst = OUT_DIR / csv.name
        shutil.copy(csv, dst)
        rows = sum(1 for _ in open(dst)) - 1
        print(f'  Restored: {csv.name}  ({rows} redova)')
    print('Restore zavrsen. Nastavi sa ćelijom 6.')

---
## SPLIT SEKCIJA — nastavak prekinutog 2-odeljnog eksperimenta

Koristiti kada imaš **djelimičan `sensitivity_2comp.csv`** (npr. N=3,5,8 gotovi)  
i hoćeš da **raspodijeliš preostale N vrednosti na oba GPU-a** da završiš brže.

**Redosljed:**
1. Preuzmi trenutni `sensitivity_2comp.csv` iz Output taba
2. Napravi novi Kaggle Dataset sa tim fajlom i priloži ga
3. Pokreni ćelije 1–4 (setup)
4. Pokreni **Split-S1** (restore djelimičnog CSV-a)
5. Pokreni **Split-S2** (paralelni run: N=10 na cuda:0, N=12 na cuda:1)
6. Pokreni **Split-S3** (merge svih CSV-ova u finalni fajl)

> Svaki GPU piše u **poseban CSV** (`split_N10/` i `split_N12/`) pa nema kolizije pri pisanju.

In [ ]:
# Split-S1 — Restore djelimičnog CSV-a u OUT_DIR
import shutil, pandas as pd
from pathlib import Path

OUT_DIR   = Path('/kaggle/working/pinn_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR = Path('/kaggle/input/pinn-checkpoint')   # <-- izmijeni naziv dataseta

src = INPUT_DIR / 'sensitivity_2comp.csv'
if not src.exists():
    print(f'Fajl nije pronađen: {src}')
else:
    dst = OUT_DIR / 'sensitivity_2comp_partial.csv'
    shutil.copy(src, dst)
    df = pd.read_csv(dst)
    print(f'Učitan djelimični CSV: {len(df)} redova')
    print(f'N vrednosti koje su gotove: {sorted(df["N"].unique())}')
    print(f'N vrednosti koje nedostaju: {[n for n in [3,5,8,10,12] if n not in df["N"].unique()]}')

In [ ]:
# Split-S2 — Paralelni run: N=10 na cuda:0, N=12 na cuda:1
# Svaki GPU piše u poseban poddirektorij — nema kolizije pri pisanju
import threading
from pathlib import Path

OUT_DIR  = Path('/kaggle/working/pinn_results')
DIR_N10  = OUT_DIR / 'split_N10'
DIR_N12  = OUT_DIR / 'split_N12'
DIR_N10.mkdir(parents=True, exist_ok=True)
DIR_N12.mkdir(parents=True, exist_ok=True)

errors = []

def _run_N10():
    try:
        print(f'[N=10] Start na {DEVICE0}')
        sens.run_sensitivity_2comp(
            test_mode=False, out_dir=DIR_N10, device=DEVICE0, n_subset=[10]
        )
        print('[N=10] ZAVRSENO')
    except Exception as e:
        errors.append(f'N10: {e}'); print(f'[N=10] GRESKA: {e}')

def _run_N12():
    try:
        print(f'[N=12] Start na {DEVICE1}')
        sens.run_sensitivity_2comp(
            test_mode=False, out_dir=DIR_N12, device=DEVICE1, n_subset=[12]
        )
        print('[N=12] ZAVRSENO')
    except Exception as e:
        errors.append(f'N12: {e}'); print(f'[N=12] GRESKA: {e}')

t1 = threading.Thread(target=_run_N10)
t2 = threading.Thread(target=_run_N12)
t1.start(); t2.start()
t1.join();  t2.join()

if errors:
    for e in errors: print(f'GRESKA: {e}')
else:
    print('Oba N zavrsena.')

In [ ]:
# Split-S3 — Merge: djelimični + N10 + N12 → finalni sensitivity_2comp.csv
import pandas as pd
from pathlib import Path

OUT_DIR = Path('/kaggle/working/pinn_results')

parts = []
for p in [
    OUT_DIR / 'sensitivity_2comp_partial.csv',
    OUT_DIR / 'split_N10' / 'sensitivity_2comp.csv',
    OUT_DIR / 'split_N12' / 'sensitivity_2comp.csv',
]:
    if p.exists():
        df = pd.read_csv(p)
        print(f'  {p.name} (iz {p.parent.name}): {len(df)} redova, N={sorted(df["N"].unique())}')
        parts.append(df)
    else:
        print(f'  NEDOSTAJE: {p}')

if parts:
    final = pd.concat(parts, ignore_index=True)
    # Ukloni duplikate po (N, sigma, seed) ako ih ima
    before = len(final)
    final  = final.drop_duplicates(subset=['N', 'sigma', 'seed']).reset_index(drop=True)
    if before != len(final):
        print(f'  Uklonjeno {before - len(final)} duplikata')
    final = final.sort_values(['N', 'sigma', 'seed']).reset_index(drop=True)
    out = OUT_DIR / 'sensitivity_2comp.csv'
    final.to_csv(out, index=False)
    print(f'\nFinalni CSV: {len(final)}/600 redova → {out}')
    ok = final[final['status'] == 'ok']
    print(f'Uspjesnih runova: {len(ok)}/{len(final)}')
    print(f'Po N: {ok["N"].value_counts().sort_index().to_dict()}')

---
## Od nule: 2-odeljni model na oba GPU-a (čist start)

Trening **samo 2-odeljnog modela** raspoređen na oba GPU-a:

| GPU | N vrednosti | Runova | Proc. trajanje |
|-----|-------------|--------|----------------|
| `cuda:0` | 3, 5, 8 | 360 | ~9h |
| `cuda:1` | 10, 12 | 240 | ~6h |

Svaki GPU piše u **poseban poddirektorij** (`2comp_gpu0/`, `2comp_gpu1/`) — nema kolizije pri pisanju.  
Na kraju ćelija **F3** merguje sve u finalni `sensitivity_2comp.csv`.

**Redosljed:** pokreni ćelije 1–4 (setup), pa F1 → F2 → F3.

In [ ]:
# F1 — Priprema: čisti output direktoriji za oba GPU streama
import shutil
from pathlib import Path

OUT_DIR  = Path('/kaggle/working/pinn_results')
DIR_GPU0 = OUT_DIR / '2comp_gpu0'   # cuda:0 → N = [3, 5, 8]
DIR_GPU1 = OUT_DIR / '2comp_gpu1'   # cuda:1 → N = [10, 12]

# Briši stare parcijalne rezultate iz prethodnih pokušaja (ako postoje)
for d in [DIR_GPU0, DIR_GPU1]:
    if d.exists():
        shutil.rmtree(d)
        print(f'  Obrisano: {d}')
    d.mkdir(parents=True, exist_ok=True)
    print(f'  Kreiran: {d}')

# Prikaži raspodjelu runova
N_GPU0 = [3, 5, 8]
N_GPU1 = [10, 12]
runs_gpu0 = len(N_GPU0) * 4 * 30
runs_gpu1 = len(N_GPU1) * 4 * 30
print(f'\ncuda:0  N={N_GPU0}  →  {runs_gpu0} runova  (~{runs_gpu0*90/3600:.0f}h @ 90s/run)')
print(f'cuda:1  N={N_GPU1}  →  {runs_gpu1} runova  (~{runs_gpu1*90/3600:.0f}h @ 90s/run)')
print(f'\nUkupno paralelno trajanje: ~{max(runs_gpu0, runs_gpu1)*90/3600:.0f}h')
print('\nSpremno. Pokreni F2.')

In [ ]:
# F2 — Paralelni trening 2-odeljnog modela: cuda:0 i cuda:1 istovremeno
# cuda:0 → N=[3,5,8] (360 runova, ~9h),  cuda:1 → N=[10,12] (240 runova, ~6h)
import threading
from pathlib import Path

OUT_DIR  = Path('/kaggle/working/pinn_results')
DIR_GPU0 = OUT_DIR / '2comp_gpu0'
DIR_GPU1 = OUT_DIR / '2comp_gpu1'

errors = []

def _run_gpu0():
    try:
        print(f'[cuda:0] Start  N=[3,5,8]')
        sens.run_sensitivity_2comp(
            test_mode=False,
            out_dir=DIR_GPU0,
            device=DEVICE0,
            n_subset=[3, 5, 8],
        )
        print('[cuda:0] ZAVRSENO')
    except Exception as e:
        errors.append(f'gpu0: {e}')
        print(f'[cuda:0] GRESKA: {e}')

def _run_gpu1():
    try:
        print(f'[cuda:1] Start  N=[10,12]')
        sens.run_sensitivity_2comp(
            test_mode=False,
            out_dir=DIR_GPU1,
            device=DEVICE1,
            n_subset=[10, 12],
        )
        print('[cuda:1] ZAVRSENO')
    except Exception as e:
        errors.append(f'gpu1: {e}')
        print(f'[cuda:1] GRESKA: {e}')

t0 = threading.Thread(target=_run_gpu0)
t1 = threading.Thread(target=_run_gpu1)
t0.start(); t1.start()
t0.join();  t1.join()

if errors:
    for e in errors: print(f'GRESKA: {e}')
else:
    print('\nOba GPU streama završena. Pokreni F3 za merge.')

In [ ]:
# F3 — Merge rezultata oba GPU-a u finalni sensitivity_2comp.csv
import pandas as pd
from pathlib import Path

OUT_DIR  = Path('/kaggle/working/pinn_results')
DIR_GPU0 = OUT_DIR / '2comp_gpu0'
DIR_GPU1 = OUT_DIR / '2comp_gpu1'
TOTAL    = 5 * 4 * 30   # 600 runova ukupno za 2-comp

parts = []
for src_dir, label in [(DIR_GPU0, 'gpu0 N=[3,5,8]'), (DIR_GPU1, 'gpu1 N=[10,12]')]:
    p = src_dir / 'sensitivity_2comp.csv'
    if p.exists():
        df = pd.read_csv(p)
        print(f'  {label}: {len(df)} redova  N={sorted(df["N"].unique())}')
        parts.append(df)
    else:
        print(f'  NEDOSTAJE: {p}')

if not parts:
    print('Nema podataka za merge — provjeri da li je F2 završen.')
else:
    final = pd.concat(parts, ignore_index=True)

    before = len(final)
    final  = final.drop_duplicates(subset=['N', 'sigma', 'seed']).reset_index(drop=True)
    if before != len(final):
        print(f'  Uklonjeno {before - len(final)} duplikata')

    final = final.sort_values(['N', 'sigma', 'seed']).reset_index(drop=True)

    out = OUT_DIR / 'sensitivity_2comp.csv'
    final.to_csv(out, index=False)

    ok = final[final['status'] == 'ok']
    print(f'\nFinalni CSV: {len(final)}/{TOTAL} redova  ({len(final)/TOTAL*100:.0f}%)')
    print(f'Uspjesnih runova: {len(ok)}/{len(final)}')
    print(f'Median PINN err_k10: {ok["pinn_err_k10"].median():.2f}%')
    print(f'Po N:')
    for n, grp in ok.groupby('N'):
        print(f'  N={n:2d}: {len(grp):3d} ok  '
              f'err_k10={grp["pinn_err_k10"].median():.1f}%  '
              f'err_V1={grp["pinn_err_V1"].median():.1f}%')
    print(f'\nSačuvano: {out}')

---
## PINN Ablation — nova inicijalizacija, isti uzorci

Koristi se kada je promijenjena inicijalizacija parametara u `pinn_model.py`.  
**Benchmark se ne pokreće ponovo** — ostaje iz originalnog `sensitivity_*.csv`.

Raspodjela po GPU-ima:

| Thread | GPU | Šta radi | Runova | Proc. trajanje |
|--------|-----|----------|--------|----------------|
| 0 | `cuda:0` | 1-comp (svi seedovi) + 2-comp seeds [0, 1] | 600 + 40 | ~5h + ~1h seq. |
| 1 | `cuda:1` | 2-comp seeds [2 … 29] | 560 | ~14h |

Svaki model piše u **poseban poddirektorij** da se izbjegne kolizija.  
Na kraju ćelija **A3** merguje 2-comp parcijalne fajlove u finalni CSV.

**Redosljed:** pokreni ćelije 1–4 (setup), pa **A1 → A2 → A3**.

In [ ]:
# A1 — Učitaj ablation modul i pripremi output direktorije
import importlib.util, shutil
from pathlib import Path

REPO_DIR  = Path('/kaggle/working/pinn-vancomycin')
OUT_DIR   = Path('/kaggle/working/pinn_results')
ABL_DIR   = OUT_DIR / 'ablation'          # root za sve ablation fajlove
ABL_1COMP = ABL_DIR / '1comp'             # 1-comp: svi seedovi, cuda:0
ABL_2A    = ABL_DIR / '2comp_seeds_0_1'  # 2-comp seedovi [0,1], cuda:0
ABL_2B    = ABL_DIR / '2comp_seeds_2_29' # 2-comp seedovi [2..29], cuda:1

for d in [ABL_1COMP, ABL_2A, ABL_2B]:
    d.mkdir(parents=True, exist_ok=True)

def _load(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

abl = _load(REPO_DIR / 'experiments' / '04_pinn_ablation.py', 'ablation')

print('Ablation modul učitan.')
print(f'  1-comp (cuda:0)           → {ABL_1COMP}')
print(f'  2-comp seeds [0,1] (cuda:0) → {ABL_2A}')
print(f'  2-comp seeds [2..29] (cuda:1) → {ABL_2B}')

SEEDS_GROUP_A = [0, 1]
SEEDS_GROUP_B = list(range(2, 30))
runs_t0 = 5*4*30 + len(SEEDS_GROUP_A)*5*4   # 1comp + 2comp seeds [0,1]
runs_t1 = len(SEEDS_GROUP_B) * 5 * 4        # 2comp seeds [2..29]
print(f'\ncuda:0: ~{runs_t0} runova  (1comp 600 + 2comp {len(SEEDS_GROUP_A)*20})')
print(f'cuda:1: ~{runs_t1} runova  (2comp seeds {SEEDS_GROUP_B[0]}..{SEEDS_GROUP_B[-1]})')
print('\nPokreni A2.')

In [ ]:
# A2 — Paralelni ablation run
#
# Thread 0 (cuda:0) — sekvencijalno:
#   1) 1-comp, svi seedovi  (600 runova, ~5h)
#   2) 2-comp, seedovi [0,1]  (40 runova, ~1h)
#
# Thread 1 (cuda:1) — paralelno s thread 0:
#   2-comp, seedovi [2..29]  (560 runova, ~14h)
#
# Svaki GPU piše u poseban poddirektorij — nema kolizije pri pisanju.

import threading

errors = []

def _thread0():
    try:
        print(f'[T0/cuda:0] 1-comp start (600 runova)')
        abl.run_ablation_1comp(
            test_mode=False,
            out_dir=ABL_1COMP,
            device=DEVICE0,
        )
        print(f'[T0/cuda:0] 1-comp ZAVRSENO — počinje 2-comp seeds {SEEDS_GROUP_A}')
        abl.run_ablation_2comp(
            test_mode=False,
            out_dir=ABL_2A,
            device=DEVICE0,
            seed_subset=SEEDS_GROUP_A,
        )
        print('[T0/cuda:0] ZAVRSENO')
    except Exception as e:
        errors.append(f'T0: {e}')
        print(f'[T0/cuda:0] GRESKA: {e}')

def _thread1():
    try:
        print(f'[T1/cuda:1] 2-comp seeds {SEEDS_GROUP_B[0]}..{SEEDS_GROUP_B[-1]} start (560 runova)')
        abl.run_ablation_2comp(
            test_mode=False,
            out_dir=ABL_2B,
            device=DEVICE1,
            seed_subset=SEEDS_GROUP_B,
        )
        print('[T1/cuda:1] ZAVRSENO')
    except Exception as e:
        errors.append(f'T1: {e}')
        print(f'[T1/cuda:1] GRESKA: {e}')

t0 = threading.Thread(target=_thread0)
t1 = threading.Thread(target=_thread1)
t0.start(); t1.start()
t0.join();  t1.join()

if errors:
    for e in errors: print(f'GRESKA: {e}')
else:
    print('\nOba threada završena. Pokreni A3 za merge 2-comp fajlova.')

In [ ]:
# A3 — Merge 2-comp ablation fajlova + prikaz rezultata
import pandas as pd
from pathlib import Path

OUT_DIR = Path('/kaggle/working/pinn_results')
ABL_DIR = OUT_DIR / 'ablation'
TOTAL   = 5 * 4 * 30   # 600 runova za 2-comp

# ── Merge 2-comp parcijala ────────────────────────────────────────────────────
parts_2comp = []
for src_dir, label in [
    (ABL_DIR / '2comp_seeds_0_1',  'seeds [0,1]'),
    (ABL_DIR / '2comp_seeds_2_29', 'seeds [2..29]'),
]:
    p = src_dir / 'sensitivity_2comp_ablation.csv'
    if p.exists():
        df = pd.read_csv(p)
        print(f'  {label}: {len(df)} redova  seeds={sorted(df["seed"].unique()[:5])}...')
        parts_2comp.append(df)
    else:
        print(f'  NEDOSTAJE: {p}')

if parts_2comp:
    final_2 = pd.concat(parts_2comp, ignore_index=True)
    before  = len(final_2)
    final_2 = final_2.drop_duplicates(subset=['N', 'sigma', 'seed']).reset_index(drop=True)
    if before != len(final_2):
        print(f'  Uklonjeno {before - len(final_2)} duplikata')
    final_2 = final_2.sort_values(['N', 'sigma', 'seed']).reset_index(drop=True)
    out_2   = ABL_DIR / 'sensitivity_2comp_ablation.csv'
    final_2.to_csv(out_2, index=False)
    ok2 = final_2[final_2['status'] == 'ok']
    print(f'\n2-comp ablation: {len(final_2)}/{TOTAL} redova  uspjesnih={len(ok2)}')
    print(f'  Median pinn_err_k10: {ok2["pinn_err_k10"].median():.2f}%')
    print(f'  Po N: {ok2["N"].value_counts().sort_index().to_dict()}')
    print(f'  Sačuvano: {out_2}')

# ── 1-comp status ─────────────────────────────────────────────────────────────
p1 = ABL_DIR / '1comp' / 'sensitivity_1comp_ablation.csv'
if p1.exists():
    df1 = pd.read_csv(p1)
    ok1 = df1[df1['status'] == 'ok']
    print(f'\n1-comp ablation: {len(df1)}/{TOTAL} redova  uspjesnih={len(ok1)}')
    print(f'  Median pinn_err_k10: {ok1["pinn_err_k10"].median():.2f}%')
    print(f'  Sačuvano: {p1}')
else:
    print(f'\n1-comp ablation: ne postoji još  ({p1})')